# 5.6 學習率怎麼隨訓練變化？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一次更新的生命週期**

忘記清零會累加；累積梯度時才刻意保留。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/backward.svg")))

**先想一想：**最早一步和最後一步，哪個應接近peak？

先用小learning rate暖身，再逐漸降低步幅，是一種配方選擇。不要一開始就把schedule和多種optimizer一起換。

**把直覺寫成數學：**warmup先線性增加；cosine decay依訓練進度平滑降低。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.training import learning_rate

rates = [learning_rate(i, 40, peak=0.001, warmup=5) for i in range(40)]
print("前6步", rates[:6], "後3步", rates[-3:])
assert max(rates) <= 0.001 + 1e-9

**如何讀結果：**排程需要固定總步數；延長總訓練時應重新說明排程，不假裝完全一樣。

**只改一件事：**只把warmup從5改成10。
